In [1]:
import os
import json
from collections import deque
from collections.abc import Mapping
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv

COLS = ["Time (s)", "Pressure (cmH2O)", "Flow (L/min)", "Volume (mL)", "Trigger", "Mode"]
CACHE_VERSION = 1
# 快取檔：欄位 -> (檔名, dtype)
CACHE_FILES = {"Time (s)": ("time.bin", np.float64), "Pressure (cmH2O)": ("pressure.bin", np.float64),
               "Flow (L/min)": ("flow.bin", np.float64), "Volume (mL)": ("volume.bin", np.float64),
               "Trigger": ("trigger.bin", np.int8), "Mode": ("mode.bin", np.int16)}

def load(csv_path, cache_dir=None):
    """
    讀錄製檔，只留 Time (s)、Trigger、Mode（100 Hz）
    P/F/V 第一次讀的時候寫到 cache_dir，之後 load 直接讀快取
    """
    if cache_dir is None:
        cache_dir = os.path.splitext(csv_path)[0] + "_cache"
    src = os.stat(csv_path)
    meta_path = os.path.join(cache_dir, "meta.json")
    meta = None
    if os.path.exists(meta_path):
        with open(meta_path, encoding="utf-8") as f:
            meta = json.load(f)
        if (meta.get("version"), meta.get("size"), meta.get("mtime")) != (CACHE_VERSION, src.st_size, src.st_mtime):
            meta = None                                            # 錄製檔換了或格式不對就重建
    if meta is None:
        meta = _build_cache(csv_path, cache_dir)
        meta.update(version=CACHE_VERSION, size=src.st_size, mtime=src.st_mtime)
        with open(meta_path, "w", encoding="utf-8") as f:          # meta 最後寫，中斷的快取不會被當成完整
            json.dump(meta, f, ensure_ascii=False)

    read = lambda c: np.fromfile(os.path.join(cache_dir, CACHE_FILES[c][0]), CACHE_FILES[c][1])
    mode = pd.Categorical.from_codes(read("Mode"), categories=meta["modes"])
    wave = pd.DataFrame({"Time (s)": read("Time (s)"), "Trigger": read("Trigger"), "Mode": mode}, copy=False)
    wave.attrs["cache_dir"] = cache_dir                            # 給 cut_breaths 找波形用
    return wave


def _iter_batches(csv_path, block=32 << 20, workers=min(8, os.cpu_count() or 1)):
    """
    解壓後切成 block 給多個執行緒解析，再照順序吐出（pyarrow 串流只有單執行緒，太慢）
    """
    stream = pa.input_stream(csv_path, compression="gzip" if csv_path.endswith(".gz") else None)
    head = stream.read(1 << 16)                                    # 標頭在第一行
    nl = head.index(b"\n") + 1
    names = head[:nl].decode("utf-8-sig").rstrip("\r\n").split(",")
    types = {c: pa.float64() for c in COLS}
    types["Mode"] = pa.string()
    ropt = pacsv.ReadOptions(column_names=names, use_threads=False, block_size=block * 2)
    copt = pacsv.ConvertOptions(include_columns=COLS, column_types=types,
                                strings_can_be_null=True)          # 空字串當空值，同 pandas
    parse = lambda buf: pacsv.read_csv(pa.py_buffer(buf), read_options=ropt, convert_options=copt)

    pending, rest = deque(), head[nl:]
    with ThreadPoolExecutor(workers) as pool:
        while True:
            data = stream.read(block)
            buf = rest + data
            cut = len(buf) if not data else buf.rfind(b"\n") + 1   # 到檔尾就整塊送出
            rest = buf[cut:]
            if cut:
                pending.append(pool.submit(parse, buf[:cut]))
            while pending and (len(pending) > workers + 2 or not data):
                yield from pending.popleft().result().to_batches()
            if not data:
                break


def _build_cache(csv_path, cache_dir):
    """平行解析錄製檔，邊讀邊寫快取"""
    os.makedirs(cache_dir, exist_ok=True)
    meta_path = os.path.join(cache_dir, "meta.json")
    if os.path.exists(meta_path):
        os.remove(meta_path)
    cats = {}                                                      # 模式字串 -> category 編號
    files = {c: open(os.path.join(cache_dir, name), "wb") for c, (name, _) in CACHE_FILES.items()}
    try:
        for batch in _iter_batches(csv_path):
            batch = batch.filter(pc.is_valid(batch.column("Pressure (cmH2O)")))   # 同原版 dropna(subset=Pressure)
            for c in COLS[:4]:
                batch.column(c).to_numpy(zero_copy_only=False).astype(np.float64).tofile(files[c])
            trig = pc.fill_null(pc.equal(batch.column("Trigger"), 1.0), False)
            trig.to_numpy(zero_copy_only=False).astype(np.int8).tofile(files["Trigger"])
            # Mode 去空白後編號，-1 = 沒有 Mode
            enc = pc.utf8_trim_whitespace(batch.column("Mode")).dictionary_encode()
            lut = np.array([cats.setdefault(v, len(cats)) for v in enc.dictionary.to_pylist()] + [-1], np.int16)
            idx = pc.fill_null(enc.indices, -1).to_numpy(zero_copy_only=False)
            lut[idx].tofile(files["Mode"])
    finally:
        for f in files.values():
            f.close()
    return {"source": os.path.abspath(csv_path), "modes": list(cats)}


FS = 100                                     # 取樣率 Hz（0.01 秒一點）


def find_breaths(wave, min_samples, max_samples):
    """
    切分規則：
    1. 兩個 trigger 之間算一口，breath_id 從 1 開始（頭尾不完整的不算）
    2. 去掉 < min_samples 的（"<0.2s"）
    3. 去掉 > max_samples 的（">10min"）
    4. 剩下的看區間內回報的 Mode
       modes   區間內出現的模式，用 " | " 串；沒回報 = "(無回報)"
       n_modes 有幾種模式，> 1 就是混到
       n_polls 有幾筆回報
    """
    t = wave["Time (s)"].to_numpy()
    idx = np.flatnonzero(wave["Trigger"].to_numpy() == 1)         # 所有 trigger 的列
    s, e = idx[:-1], idx[1:]                                       # 這口的列範圍 [s, e)
    n = e - s                                                      # 這口的樣本點數

    excluded = np.select([n < min_samples, n > max_samples], ["<0.2s", ">10min"], "")
    keep = excluded == ""

    # 慢數據只有輪詢到的列有 Mode，-1 = 沒回報
    codes = wave["Mode"].cat.codes.to_numpy()
    cats = np.asarray(wave["Mode"].cat.categories, dtype=object)
    poll = np.flatnonzero(codes >= 0)                              # 有回報的列
    k = np.searchsorted(s, poll, side="right") - 1                 # 每筆回報落在第幾口
    inside = (k >= 0) & (poll < e[np.maximum(k, 0)]) & keep[np.maximum(k, 0)]
    polls = pd.DataFrame({"k": k[inside], "mode": cats[codes[poll[inside]]]})

    g = polls.groupby("k", sort=False)["mode"]
    modes = pd.Series("", index=range(len(s)), dtype=object)
    modes[keep] = "(無回報)"
    joined = g.agg(lambda m: " | ".join(pd.unique(m)))
    modes[joined.index] = joined
    n_modes = np.zeros(len(s), int)
    n_modes[keep] = 0
    nm = g.nunique()
    n_modes[nm.index.to_numpy()] = nm.to_numpy()
    n_polls = np.zeros(len(s), int)
    cnt = g.size()
    n_polls[cnt.index.to_numpy()] = cnt.to_numpy()

    return pd.DataFrame({"breath_id": np.arange(1, len(s) + 1), "n_samples": n,
                         "duration_s": np.round(n / FS, 2),
                         "start_time": t[s], "end_time": t[e - 1],
                         "excluded": excluded, "modes": modes.to_numpy(),
                         "n_modes": n_modes, "n_polls": n_polls})


def cut_breaths(wave, breaths):
    """
    照 breaths 的起訖時間從 wave 剪出每口波形，回傳 BreathWaves（當 dict 用）
    """
    t = wave["Time (s)"].to_numpy()
    # 二分搜尋一次找出所有口的起訖列
    # ±1e-9 怕浮點誤差漏掉頭尾
    s = np.searchsorted(t, breaths["start_time"].to_numpy() - 1e-9)                 # 起點列
    e = np.searchsorted(t, breaths["end_time"].to_numpy() + 1e-9, side="right")     # 終點列的下一列
    return BreathWaves(wave.attrs["cache_dir"], breaths["breath_id"].to_numpy(), s, np.maximum(e, s))


class BreathWaves(Mapping):
    """硬碟快取上的呼吸波形，當 dict 用"""
    def __init__(self, cache_dir, ids, starts, ends):
        self.ids, self.starts, self.ends = ids, starts, ends
        self._pos = {int(b): k for k, b in enumerate(ids)}
        self._data = {c: np.memmap(os.path.join(cache_dir, name), dt, mode="r")
                      for c, (name, dt) in CACHE_FILES.items() if c in COLS[:4]}

    def __getitem__(self, bid):
        k = self._pos[bid]
        a, b = self.starts[k], self.ends[k]
        return pd.DataFrame({c: np.array(v[a:b]) for c, v in self._data.items()})

    def __iter__(self):
        return (int(b) for b in self.ids)

    def __len__(self):
        return len(self.ids)


# 切分檔：存、讀、合併
SIG = ["Pressure (cmH2O)", "Flow (L/min)", "Volume (mL)"]       # 切分檔的通道順序
META = ["breath_id", "n_samples", "duration_s", "start_time", "end_time", "modes", "n_modes", "n_polls"]


def save_split(wave, breaths, out_path, rec_name, chunk=10_000):
    """
    把 breaths 的波形存成一個 .npz
    所有口接成一條長陣列，第 i 口 = signal[offsets[i]:offsets[i+1]]
    """
    t = wave["Time (s)"].to_numpy()
    s = np.searchsorted(t, breaths["start_time"].to_numpy() - 1e-9)
    e = np.searchsorted(t, breaths["end_time"].to_numpy() + 1e-9, side="right")
    n = e - s
    offsets = np.concatenate([[0], np.cumsum(n)]).astype(np.int64)
    mm = {c: np.memmap(os.path.join(wave.attrs["cache_dir"], CACHE_FILES[c][0]), CACHE_FILES[c][1], mode="r")
          for c in SIG}
    sig = {c: np.empty(offsets[-1], np.float32) for c in SIG}
    for a in range(0, len(s), chunk):                              # 分批取
        ss, nn = s[a:a + chunk], n[a:a + chunk]
        idx = np.repeat(ss - np.concatenate([[0], np.cumsum(nn)[:-1]]), nn) + np.arange(nn.sum())
        for c in SIG:
            sig[c][offsets[a]:offsets[a] + len(idx)] = mm[c][idx]
    meta = {k: breaths[k].to_numpy() for k in META}
    meta["modes"] = meta["modes"].astype(str)
    meta["rec"] = np.full(len(s), rec_name)
    np.savez(out_path, offsets=offsets, **{f"sig_{i}": sig[c] for i, c in enumerate(SIG)},
             **{f"meta_{k}": v for k, v in meta.items()})


class Split:
    """
    讀切分檔（單檔或合併後的都可以）
      ds[i]        第 i 口，shape (3, 點數)
      ds.meta      每口的資訊
      ds.select(m) 篩選，例如 ds.select(ds.meta["modes"] == "PC-AC")
    """
    def __init__(self, path=None, sig=None, offsets=None, meta=None):
        if path is not None:
            z = np.load(path)
            sig = np.stack([z[f"sig_{i}"] for i in range(len(SIG))])
            offsets = z["offsets"]
            meta = pd.DataFrame({k[5:]: z[k] for k in z.files if k.startswith("meta_")})
        self.sig, self.offsets, self.meta = sig, offsets, meta.reset_index(drop=True)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, i):
        return self.sig[:, self.offsets[i]:self.offsets[i + 1]]

    def select(self, mask):
        keep = np.flatnonzero(np.asarray(mask))
        parts = [self.sig[:, self.offsets[i]:self.offsets[i + 1]] for i in keep]
        n = np.array([p.shape[1] for p in parts], np.int64)
        sig = np.concatenate(parts, axis=1) if parts else np.empty((len(SIG), 0), np.float32)
        return Split(sig=sig, offsets=np.concatenate([[0], np.cumsum(n)]), meta=self.meta.iloc[keep])

    def save(self, out_path):
        meta = {k: self.meta[k].to_numpy() for k in self.meta}
        for k in ("modes", "rec"):
            meta[k] = meta[k].astype(str)
        np.savez(out_path, offsets=self.offsets, **{f"sig_{i}": self.sig[i] for i in range(len(SIG))},
                 **{f"meta_{k}": v for k, v in meta.items()})


def merge_splits(paths):
    """多個切分檔接成一個，用 rec 欄位分是哪個錄製檔"""
    parts = [Split(p) for p in paths]
    recs = [set(p.meta["rec"]) for p in parts]
    seen = set()
    for p, r in zip(paths, recs):
        if seen & r:
            raise ValueError(f"{p} 的錄製檔 {sorted(seen & r)} 已經合併過，重複了")
        seen |= r
    shift = np.cumsum([0] + [p.offsets[-1] for p in parts[:-1]])
    offsets = np.concatenate([parts[0].offsets[:1]] + [p.offsets[1:] + d for p, d in zip(parts, shift)])
    return Split(sig=np.concatenate([p.sig for p in parts], axis=1), offsets=offsets,
                 meta=pd.concat([p.meta for p in parts], ignore_index=True))


def to_fixed(ds, length, method="resample"):
    """
    每口轉成固定長度，回傳 X (口數, 3, length)、lengths（原本點數）
      "resample"：線性內插成 length 點
      "pad"     ：取前 length 點，不夠補 0；mask 標哪些是真的資料
    """
    N = len(ds)
    X = np.zeros((N, len(SIG), length), np.float32)
    lengths = np.diff(ds.offsets).astype(np.int64)
    if method == "resample":
        grid = np.linspace(0, 1, length)
        for i in range(N):
            x = ds[i]
            src = np.linspace(0, 1, x.shape[1])
            for c in range(len(SIG)):
                X[i, c] = np.interp(grid, src, x[c])
        return X, lengths
    if method == "pad":
        mask = np.zeros((N, length), bool)
        for i in range(N):
            x = ds[i][:, :length]
            X[i, :, :x.shape[1]] = x
            mask[i, :x.shape[1]] = True
        return X, lengths, mask
    raise ValueError("method 只能是 'resample' 或 'pad'")


# 切分設定

In [2]:
path = r"D:\pva_training\data"              # 錄製檔資料夾
REC = "5.gz"                                     # 錄製檔名稱
MIN_SAMPLES = int(round(0.2 * FS))               # 20 點 = 0.2 秒，少於就去掉
MAX_SAMPLES = int(round(10 * 60 * FS))           # 60000 點 = 10 分鐘，多於就去掉
SAVE_CSV = True                                  # 每口結果存 csv
SPLIT_DIR = os.path.join(path, "splits")         # 切分檔存這裡，檔名 錄製檔名_split.npz

In [3]:
# 1. 讀錄製檔，第二次之後讀快取
wave = load(os.path.join(path, REC))
print(f"共 {len(wave):,} 點，模式：{list(wave['Mode'].cat.categories)}")
print("----------------------------------")

# 2. 切口、去掉太短太長的、看 Mode
breaths = find_breaths(wave, MIN_SAMPLES, MAX_SAMPLES)
kept = breaths[breaths["excluded"] == ""]
print(f"trigger 切出 {len(breaths)} 口")
print(f"  排除 <0.2s ：{(breaths['excluded'] == '<0.2s').sum()} 口")
print(f"  排除 >10min：{(breaths['excluded'] == '>10min').sum()} 口")
print(f"  剩下       ：{len(kept)} 口")
pc = kept[kept["modes"] == "PC-AC"]                  # 只算純 PC-AC
if len(pc):
    print(f"  PC-AC 長度 ：{pc['n_samples'].min()} ~ {pc['n_samples'].max()} 點"
          f"（{pc['duration_s'].min():.2f} ~ {pc['duration_s'].max():.2f} 秒），"
          f"中位數 {int(pc['n_samples'].median())} 點，共 {len(pc)} 口")
else:
    print("  PC-AC 長度 ：沒有 PC-AC 的口")
print("----------------------------------")

# 3. 剩下的口各是什麼模式
print("區間內的模式：")
print(kept["modes"].value_counts().to_string())
print("----------------------------------")

# 4. 有沒有一口混到兩種以上模式
dup = kept[kept["n_modes"] > 1]
print(f"重複：{len(dup)} 口")
if len(dup):
    print(dup.to_string(max_rows=40))
print("----------------------------------")

if SAVE_CSV:
    out = os.path.join(path, f"{os.path.splitext(REC)[0]}_breaths.csv")
    breaths.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"每口結果存到 {out}")

# 5. 存切分檔，模式全存，之後再挑
os.makedirs(SPLIT_DIR, exist_ok=True)
split_path = os.path.join(SPLIT_DIR, f"{os.path.splitext(REC)[0]}_split.npz")
save_split(wave, kept, split_path, rec_name=REC)
print(f"切分檔存到 {split_path}（{len(kept)} 口）")

共 300,245,322 點，模式：['PC-AC', 'SPN-CPAP', 'VC-AC']
----------------------------------
trigger 切出 547691 口
  排除 <0.2s ：152 口
  排除 >10min：25 口
  剩下       ：547514 口
  PC-AC 長度 ：77 ~ 13417 點（0.77 ~ 134.17 秒），中位數 423 點，共 545157 口
----------------------------------
區間內的模式：
modes
PC-AC               545157
(無回報)                 2236
SPN-CPAP                65
VC-AC                   50
PC-AC | SPN-CPAP         6
----------------------------------
重複：6 口
        breath_id  n_samples  duration_s  start_time    end_time excluded             modes  n_modes  n_polls
13096       13097       3725       37.25    40683.15    40720.39           PC-AC | SPN-CPAP        2        7
38613       38614      13193      131.93   157138.36   157270.28           PC-AC | SPN-CPAP        2       23
69186       69187       2531       25.31   393617.87   393643.17           PC-AC | SPN-CPAP        2        5
77852       77853       4618       46.18   449733.12   449779.29           PC-AC | SPN-CPAP        2        8
